# Lab 25-02 · Aerial Images, Focus, Photon Noise and Scanner Economics

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import litho as L

## Part 1 · Resolution cutoff

*Predict* the pitch at which a 0.33-NA EUV image vanishes with an ideal dipole.

In [2]:
lam, na = 13.5, 0.33
pitches = np.linspace(18, 30, 241)
c = [L.image_contrast(p, lam, na, poles=L.dipole_for(p, lam)) if lam / (2 * p) <= na else 0.0 for p in pitches]
cut = min(p for p, ci in zip(pitches, c) if ci > 0)
print(f"EUV cutoff with an ideal dipole: {cut:.2f} nm (theory {lam/(2*na):.2f} nm)")
pa = np.linspace(60, 200, 281)
on = [L.image_contrast(p, 193, 1.35, 1.44) for p in pa]
dip = [L.image_contrast(p, 193, 1.35, 1.44, poles=L.dipole_for(p, 193)) if 193 / (2 * p) <= 1.35 else 0.0 for p in pa]
plt.figure()
plt.plot(pa, on, label="on-axis"); plt.plot(pa, dip, label="ideal dipole")
plt.xlabel("pitch (nm)"); plt.ylabel("image contrast"); plt.legend(); plt.grid(True)
plt.show()

EUV cutoff with an ideal dipole: 20.50 nm (theory 20.45 nm)


## Part 2 · Focus

*Predict* how 50 nm of defocus changes a 160 nm pitch's contrast with on-axis and with dipole illumination.

In [3]:
x = np.linspace(0, 320, 1001)
plt.figure()
for poles, name in (((0.0,), "on-axis"), (L.dipole_for(160, 193), "dipole")):
    for z, ls in ((0, "-"), (50, "--")):
        I = L.aerial_image(x, 160, 193, 1.35, 1.44, poles=poles, defocus=z)
        print(f"{name:8s} defocus {z:3d} nm: contrast {L.contrast(I):.3f}")
        plt.plot(x, I, ls, label=f"{name}, z = {z} nm")
plt.xlabel("x (nm)"); plt.ylabel("intensity (a.u.)"); plt.legend(fontsize=7); plt.grid(True)
plt.show()

on-axis  defocus   0 nm: contrast 1.000
on-axis  defocus  50 nm: contrast 0.669
dipole   defocus   0 nm: contrast 0.906
dipole   defocus  50 nm: contrast 0.906


## Part 3 · Photon noise

*Predict* the standard deviation of absorbed photons in a 10 nm × 10 nm contact at 30 mJ/cm² with 20% absorption.

In [4]:
a = L.simulate_absorbed(30, 13.5, 100, 0.2)
n_mean = L.photons_per_nm2(30, 13.5) * 100 * 0.2
print(f"mean {a.mean():.1f} (expected {n_mean:.1f}); std {a.std():.2f} (sqrt N = {np.sqrt(n_mean):.2f})")
plt.figure()
plt.hist(a, bins=60)
plt.xlabel("absorbed photons per contact"); plt.ylabel("count"); plt.grid(True)
plt.show()

mean 407.8 (expected 407.8); std 20.25 (sqrt N = 20.19)


## Part 4 · Dose and throughput

*Predict* the throughput at 60 mJ/cm² for a scanner giving 220 wph at 30 mJ/cm² with 6 s of overhead.

In [5]:
doses = np.linspace(10, 120, 111)
w = [L.throughput_wph(220, 30, 6.0, d) for d in doses]
print(f"throughput at 60 mJ/cm2: {L.throughput_wph(220, 30, 6.0, 60):.1f} wph")
plt.figure()
plt.plot(doses, w)
plt.xlabel("dose (mJ/cm2)"); plt.ylabel("wafers per hour"); plt.grid(True)
plt.show()

throughput at 60 mJ/cm2: 134.7 wph
